# Analyse des Crashs d'Avion (1908 – 2023)

**Dataset :** Airplane Crashes and Fatalities Since 1908 to 2023  
**Outils :** Pandas, NumPy, SciPy, Matplotlib, Seaborn

---
### Plan
1. Import & nettoyage des données
2. Analyse exploratoire (EDA)
3. Analyse statistique avec SciPy
4. Visualisations
5. Résumé des insights

## 1. Import des librairies et chargement des données

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

%matplotlib inline
sns.set_theme(style='whitegrid')

In [ ]:
df = pd.read_csv('Airplane_Crashes_and_Fatalities_Since_1908_t0_2023.csv')

print('Dimensions du dataset :', df.shape)
print('\nColonnes :', df.columns.tolist())
df.head()

## 2. Nettoyage et préparation des données

In [ ]:
# Vérifier les valeurs manquantes
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)

missing_summary = pd.DataFrame({
    'Valeurs manquantes': missing,
    'Pourcentage (%)': missing_pct
})
print(missing_summary)

In [ ]:
# Convertir la colonne Date en format datetime
df['Date'] = pd.to_datetime(df['Date'], errors='coerce')

# Extraire l'année, le mois et la décennie
df['Year']   = df['Date'].dt.year
df['Month']  = df['Date'].dt.month
df['Decade'] = (df['Year'] // 10) * 10

# Renommer les colonnes pour plus de clarté
df = df.rename(columns={
    'Aboard':                'total_aboard',
    'Fatalities':            'fatalities',
    'Ground':                'ground_deaths',
    'Aboard Passangers':     'passengers_aboard',
    'Aboard Crew':           'crew_aboard',
    'Fatalities Passangers': 'passenger_fatalities',
    'Fatalities Crew':       'crew_fatalities'
})

# Remplir les valeurs manquantes numériques par 0
numeric_cols = ['total_aboard', 'fatalities', 'ground_deaths',
                'passengers_aboard', 'crew_aboard',
                'passenger_fatalities', 'crew_fatalities']

df[numeric_cols] = df[numeric_cols].fillna(0)

# Calculer les survivants et le taux de survie
df['survivors']    = (df['total_aboard'] - df['fatalities']).clip(lower=0)
df['survival_rate'] = np.where(
    df['total_aboard'] > 0,
    (df['survivors'] / df['total_aboard'] * 100).round(2),
    np.nan
)

print('Nettoyage terminé.')
df[['Date', 'Year', 'Decade', 'total_aboard', 'fatalities', 'survivors', 'survival_rate']].head()

## 3. Analyse Exploratoire (EDA)

In [ ]:
# Statistiques générales
total_crashes    = len(df)
total_fatalities = df['fatalities'].sum()
total_aboard     = df['total_aboard'].sum()
total_survivors  = df['survivors'].sum()
overall_survival = (total_survivors / total_aboard * 100).round(2)

print(f'Nombre total de crashs          : {total_crashes:,}')
print(f'Total de personnes à bord       : {int(total_aboard):,}')
print(f'Total de morts                  : {int(total_fatalities):,}')
print(f'Total de survivants             : {int(total_survivors):,}')
print(f'Taux de survie global           : {overall_survival}%')

In [ ]:
# Crashs et morts par année
crashes_by_year    = df.groupby('Year').size().reset_index(name='nb_crashes')
fatalities_by_year = df.groupby('Year')['fatalities'].sum().reset_index()

print('Top 10 années avec le plus de crashs :')
print(crashes_by_year.sort_values('nb_crashes', ascending=False).head(10))

In [ ]:
# Crashs, morts et taux de survie par décennie
crashes_by_decade    = df.groupby('Decade').size().reset_index(name='nb_crashes')
fatalities_by_decade = df.groupby('Decade')['fatalities'].sum().reset_index()
survival_by_decade   = df.groupby('Decade')['survival_rate'].mean().round(2).reset_index()

decade_summary = crashes_by_decade.merge(fatalities_by_decade, on='Decade')
decade_summary = decade_summary.merge(survival_by_decade, on='Decade')
decade_summary.columns = ['Décennie', 'Nb crashs', 'Total morts', 'Taux survie moyen (%)']
print(decade_summary.to_string(index=False))

In [ ]:
# Top 10 des opérateurs avec le plus de crashs
top_operators = df['Operator'].value_counts().head(10)
print('Top 10 opérateurs les plus accidentés :')
print(top_operators)

In [ ]:
# Extraire le pays depuis la colonne Location
df['country'] = df['Location'].dropna().str.split(',').str[-1].str.strip()

top_countries = df['country'].value_counts().head(10)
print('Top 10 pays avec le plus de crashs :')
print(top_countries)

## 4. Analyse Statistique avec NumPy et SciPy

In [ ]:
# Statistiques descriptives sur les fatalités
fat = df['fatalities'].dropna()

mean_fat   = np.mean(fat)
median_fat = np.median(fat)
std_fat    = np.std(fat)
skew_fat   = stats.skew(fat)
kurt_fat   = stats.kurtosis(fat)

print('--- Fatalités par crash ---')
print(f'Moyenne        : {mean_fat:.2f}')
print(f'Médiane        : {median_fat:.2f}')
print(f'Écart-type     : {std_fat:.2f}')
print(f'Skewness       : {skew_fat:.2f}  (>0 = asymétrie à droite)')
print(f'Kurtosis       : {kurt_fat:.2f}  (>3 = queues lourdes)')

In [ ]:
# Statistiques sur le taux de survie
surv = df['survival_rate'].dropna()

print('--- Taux de survie (%) ---')
print(f'Moyenne        : {np.mean(surv):.2f}%')
print(f'Médiane        : {np.median(surv):.2f}%')
print(f'Écart-type     : {np.std(surv):.2f}%')
print(f'Min            : {surv.min():.2f}%')
print(f'Max            : {surv.max():.2f}%')

In [ ]:
# Test d'hypothèse : les crashs avant 1970 ont-ils un nombre de morts différent d'après 1970 ?
# H0 : pas de différence significative
# H1 : il y a une différence significative

before_1970 = df[df['Year'] < 1970]['fatalities'].dropna()
after_1970  = df[df['Year'] >= 1970]['fatalities'].dropna()

# Test de Mann-Whitney (non paramétrique, adapté aux distributions asymétriques)
stat, p_value = stats.mannwhitneyu(before_1970, after_1970, alternative='two-sided')

print('Test de Mann-Whitney : morts avant 1970 vs après 1970')
print(f'Moyenne avant 1970  : {before_1970.mean():.2f}')
print(f'Moyenne après 1970  : {after_1970.mean():.2f}')
print(f'Statistique U       : {stat:.2f}')
print(f'p-value             : {p_value:.4f}')

if p_value < 0.05:
    print('\n→ Différence SIGNIFICATIVE (p < 0.05) : on rejette H0')
else:
    print('\n→ Pas de différence significative (p >= 0.05) : on garde H0')

In [ ]:
# Corrélation : est-ce que le nombre de crashs diminue avec les années ?
crashes_per_year = df.groupby('Year').size()

corr, p_corr = stats.pearsonr(crashes_per_year.index, crashes_per_year.values)

print('Corrélation de Pearson : Année vs Nombre de crashs')
print(f'Corrélation r  : {corr:.4f}')
print(f'p-value        : {p_corr:.4f}')

if corr > 0:
    print('\n→ Les crashs ont AUGMENTÉ avec le temps (r > 0)')
else:
    print('\n→ Les crashs ont DIMINUÉ avec le temps (r < 0)')

## 5. Visualisations

In [ ]:
# Évolution des crashs et des morts dans le temps
fig, axes = plt.subplots(2, 1, figsize=(14, 10))

axes[0].plot(crashes_by_year['Year'], crashes_by_year['nb_crashes'],
             color='steelblue', linewidth=1.5)
axes[0].fill_between(crashes_by_year['Year'], crashes_by_year['nb_crashes'],
                     alpha=0.3, color='steelblue')
axes[0].set_title('Nombre de crashs par année (1908–2023)', fontsize=13)
axes[0].set_xlabel('Année')
axes[0].set_ylabel('Nombre de crashs')

axes[1].plot(fatalities_by_year['Year'], fatalities_by_year['fatalities'],
             color='crimson', linewidth=1.5)
axes[1].fill_between(fatalities_by_year['Year'], fatalities_by_year['fatalities'],
                     alpha=0.3, color='crimson')
axes[1].set_title('Nombre de morts par année (1908–2023)', fontsize=13)
axes[1].set_xlabel('Année')
axes[1].set_ylabel('Nombre de morts')

plt.tight_layout()
plt.show()

In [ ]:
# Crashs et taux de survie par décennie
valid_decades = decade_summary[decade_summary['Décennie'] >= 1910]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].bar(valid_decades['Décennie'].astype(str),
            valid_decades['Nb crashs'],
            color='steelblue', edgecolor='white')
axes[0].set_title('Nombre de crashs par décennie', fontsize=13)
axes[0].set_xlabel('Décennie')
axes[0].set_ylabel('Nombre de crashs')
axes[0].tick_params(axis='x', rotation=45)

axes[1].plot(valid_decades['Décennie'].astype(str),
             valid_decades['Taux survie moyen (%)'],
             marker='o', color='green', linewidth=2)
axes[1].set_title('Taux de survie moyen par décennie', fontsize=13)
axes[1].set_xlabel('Décennie')
axes[1].set_ylabel('Taux de survie (%)')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

In [ ]:
# Distribution des fatalités et du taux de survie
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(df['fatalities'].dropna(), bins=50, color='crimson', edgecolor='white', alpha=0.8)
axes[0].axvline(mean_fat,   color='blue',   linestyle='--', label=f'Moyenne : {mean_fat:.1f}')
axes[0].axvline(median_fat, color='orange', linestyle='--', label=f'Médiane : {median_fat:.1f}')
axes[0].set_title('Distribution des morts par crash', fontsize=13)
axes[0].set_xlabel('Nombre de morts')
axes[0].set_ylabel('Fréquence')
axes[0].legend()

axes[1].hist(df['survival_rate'].dropna(), bins=40, color='green', edgecolor='white', alpha=0.8)
axes[1].set_title('Distribution du taux de survie (%)', fontsize=13)
axes[1].set_xlabel('Taux de survie (%)')
axes[1].set_ylabel('Fréquence')

plt.tight_layout()
plt.show()

In [ ]:
# Top 10 opérateurs les plus accidentés
top_op_df = top_operators.reset_index()
top_op_df.columns = ['Operator', 'nb_crashes']

plt.figure(figsize=(12, 5))
sns.barplot(data=top_op_df, x='nb_crashes', y='Operator', palette='Blues_r')
plt.title('Top 10 des opérateurs avec le plus de crashs', fontsize=13)
plt.xlabel('Nombre de crashs')
plt.ylabel('')
plt.tight_layout()
plt.show()

In [ ]:
# Top 10 pays les plus touchés
top_ct_df = top_countries.reset_index()
top_ct_df.columns = ['Pays', 'nb_crashes']

plt.figure(figsize=(12, 5))
sns.barplot(data=top_ct_df, x='nb_crashes', y='Pays', palette='Reds_r')
plt.title('Top 10 des pays avec le plus de crashs', fontsize=13)
plt.xlabel('Nombre de crashs')
plt.ylabel('')
plt.tight_layout()
plt.show()

In [ ]:
# Crashs par mois (saisonnalité)
crashes_by_month = df.groupby('Month').size().reset_index(name='nb_crashes')
month_names = ['Jan', 'Fév', 'Mar', 'Avr', 'Mai', 'Juin',
               'Juil', 'Août', 'Sep', 'Oct', 'Nov', 'Déc']
crashes_by_month['month_name'] = crashes_by_month['Month'].apply(
    lambda m: month_names[m - 1] if 1 <= m <= 12 else 'Inconnu'
)

plt.figure(figsize=(10, 4))
sns.barplot(data=crashes_by_month, x='month_name', y='nb_crashes', color='steelblue')
plt.title('Nombre de crashs par mois', fontsize=13)
plt.xlabel('Mois')
plt.ylabel('Nombre de crashs')
plt.tight_layout()
plt.show()

In [ ]:
# Boxplot avant vs après 1970 (illustre le test Mann-Whitney)
df['periode'] = df['Year'].apply(lambda y: 'Avant 1970' if y < 1970 else 'Après 1970')

plt.figure(figsize=(8, 5))
sns.boxplot(data=df, x='periode', y='fatalities',
            palette=['#e74c3c', '#2ecc71'],
            order=['Avant 1970', 'Après 1970'],
            showfliers=False)
plt.title('Fatalités par crash : avant vs après 1970', fontsize=13)
plt.xlabel('')
plt.ylabel('Nombre de morts')
plt.tight_layout()
plt.show()

## 6. Résumé des Insights

### Ce que les données nous montrent :

**1. Tendance générale des crashs**  
Le nombre de crashs a fortement augmenté entre 1908 et les années 1970 (expansion de l'aviation commerciale), puis a diminué grâce aux améliorations technologiques et aux régulations de sécurité.

**2. Taux de survie**  
Le taux de survie global reste faible. La médiane souvent proche de 0% montre que beaucoup de crashs sont fatals à 100%. C'est pourquoi la moyenne et la médiane divergent fortement.

**3. Test statistique (Mann-Whitney)**  
Le test montre s'il existe une différence significative dans les fatalités avant et après 1970.  
Les crashs après 1970 impliquent plus de personnes (avions plus grands), mais les progrès en sécurité ont compensé.

**4. Corrélation temporelle**  
La corrélation de Pearson entre l'année et le nombre de crashs indique si la tendance est à la hausse ou à la baisse sur l'ensemble de la période.

**5. Opérateurs et pays**  
Aeroflot (aviation soviétique) domine le classement des opérateurs accidentés. Les USA arrivent en tête des pays, ce qui est lié à leur volume très élevé de trafic aérien historique.

**6. Distribution des morts**  
La distribution est très asymétrique vers la droite (skewness élevé) : la majorité des crashs tuent peu de personnes, mais quelques catastrophes majeures ont des bilans très lourds.

---
**Conclusion :** L'aviation est devenue beaucoup plus sûre au fil du temps. Malgré l'augmentation du nombre d'avions et de passagers, le nombre de crashs et de morts a globalement diminué depuis les années 1970 grâce aux avancées technologiques et aux normes de sécurité.